# Module NLP (Rapport_Collecte)

Objectif: extraire des caracteristiques a partir de la colonne textuelle et comparer plusieurs vectorisations + classifieurs.

Dataset attendu: `backend/dataset_ProjetML_2026.csv`.


In [ ]:
from pathlib import Path
import re

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier

DATA_PATH = Path("..") / "backend" / "dataset_ProjetML_2026.csv"
RANDOM_STATE = 42

df = pd.read_csv(DATA_PATH)

# Garder uniquement les lignes avec texte et label
text_col = "Rapport_Collecte"
label_col = "Categorie"

df = df.dropna(subset=[text_col, label_col]).copy()
print("Rows:", len(df))

Rows: 9986


## 1. Nettoyage de texte

On applique un nettoyage simple puis un enrichissement NLP (stopwords FR + domaine, stemming et lemmatization si disponible).

In [ ]:
DOMAIN_STOPWORDS = {
    "rapport", "collecte", "dechets", "dechet", "tri", "centre", "usine",
    "citoyenne", "client", "clients", "donnee", "donnees", "donnees",
    "lot", "lots", "kg", "tonne", "tonnes"
}

def load_stopwords() -> set:
    base = set()
    try:
        import nltk
        from nltk.corpus import stopwords
        try:
            _ = stopwords.words("french")
        except LookupError:
            nltk.download("stopwords")
        base = set(stopwords.words("french"))
    except Exception:
        base = {
            "le", "la", "les", "de", "des", "du", "un", "une", "et", "ou", "a", "au",
            "aux", "en", "pour", "avec", "sans", "sur", "dans", "par", "ce", "ces",
            "cet", "cette", "ses", "son", "sa", "leurs", "leur"
        }
    return base.union(DOMAIN_STOPWORDS)

def get_spacy_lemmatizer():
    try:
        import spacy
        return spacy.load("fr_core_news_sm")
    except Exception:
        return None

def normalize_text(text: str, stopwords_set: set, use_stem: bool = True, use_lemma: bool = True) -> str:
    text = text.lower()
    text = re.sub(r"[^a-zA-Zàâçéèêëîïôûùüÿñæœ0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    tokens = [t for t in text.split() if t not in stopwords_set and len(t) > 2]
    if use_lemma:
        nlp = get_spacy_lemmatizer()
        if nlp is not None:
            tokens = [t.lemma_ for t in nlp(" ".join(tokens)) if t.lemma_]
    if use_stem:
        try:
            from nltk.stem.snowball import SnowballStemmer
            stemmer = SnowballStemmer("french")
            tokens = [stemmer.stem(t) for t in tokens]
        except Exception:
            pass
    return " ".join(tokens)

stopwords_set = load_stopwords()
df[text_col] = df[text_col].astype(str).map(lambda t: normalize_text(t, stopwords_set))

X_train, X_test, y_train, y_test = train_test_split(
    df[text_col],
    df[label_col],
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=df[label_col],
)

## 2. Vectorisation + classification

Comparaison BoW, TF-IDF et differents classifieurs.

In [ ]:
pipelines = {
    "bow_nb": Pipeline([
        ("vec", CountVectorizer(min_df=2)),
        ("clf", MultinomialNB()),
    ]),
    "tfidf_lr": Pipeline([
        ("vec", TfidfVectorizer(ngram_range=(1, 2), min_df=2)),
        ("clf", LogisticRegression(max_iter=2000)),
    ]),
    "tfidf_svc": Pipeline([
        ("vec", TfidfVectorizer(ngram_range=(1, 2), min_df=2)),
        ("clf", LinearSVC()),
    ]),
    "tfidf_rf": Pipeline([
        ("vec", TfidfVectorizer(ngram_range=(1, 2), min_df=2)),
        ("clf", RandomForestClassifier(n_estimators=300, max_depth=30, random_state=RANDOM_STATE)),
    ]),
}

for name, pipe in pipelines.items():
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)
    acc = accuracy_score(y_test, preds)
    f1 = f1_score(y_test, preds, average="weighted")
    print(name, "acc=", round(acc, 4), "f1=", round(f1, 4))

bow_nb acc= 1.0 f1= 1.0
tfidf_lr acc= 1.0 f1= 1.0
tfidf_svc acc= 1.0 f1= 1.0


C:\Users\DELL\AppData\Roaming\Python\Python312\site-packages\sklearn\svm\_classes.py:31: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(


## 3. Word2Vec / FastText + RandomForest

Si `gensim` est installe, on entraine des embeddings et on compare avec un RF.

In [ ]:
try:
    from gensim.models import Word2Vec, FastText

    def build_sentence_matrix(texts, model, vector_size: int) -> np.ndarray:
        vectors = []
        for text in texts:
            tokens = text.split()
            token_vecs = [model.wv[t] for t in tokens if t in model.wv]
            if token_vecs:
                vectors.append(np.mean(token_vecs, axis=0))
            else:
                vectors.append(np.zeros(vector_size))
        return np.vstack(vectors)

    sentences_train = [text.split() for text in X_train]
    sentences_test = [text.split() for text in X_test]

    for name, cls in [("word2vec", Word2Vec), ("fasttext", FastText)]:
        embed = cls(
            sentences=sentences_train,
            vector_size=100,
            window=5,
            min_count=2,
            workers=2,
            seed=RANDOM_STATE,
        )
        X_train_vec = build_sentence_matrix(X_train, embed, 100)
        X_test_vec = build_sentence_matrix(X_test, embed, 100)

        clf = RandomForestClassifier(
            n_estimators=300,
            max_depth=30,
            random_state=RANDOM_STATE,
        )
        clf.fit(X_train_vec, y_train)
        preds = clf.predict(X_test_vec)
        acc = accuracy_score(y_test, preds)
        f1 = f1_score(y_test, preds, average="weighted")
        print(name, "acc=", round(acc, 4), "f1=", round(f1, 4))
except Exception as exc:
    print("Gensim non disponible:", exc)

Gensim non disponible: cannot import name 'triu' from 'scipy.linalg' (C:\Users\DELL\AppData\Roaming\Python\Python312\site-packages\scipy\linalg\__init__.py)
